# Engenharia de Freatures: The Movies Dataset
Esta etapa é a mais complexa do pipeline de dados de filmes. Ela transforma colunas armazenadas em formato JSON serializado em estruturas Python nativas, aplica uma **regra de negócio baseada no Princípio de Pareto** para correção de franquias ausentes, e cria features derivadas que habilitam análises comparativas entre filmes independentes e propriedades intelectuais de franquia.

- **Escopo:** Deserialização de colunas JSON, correção de franquias por mapeamento explícito e criação de features binárias e de granularidade
- **Produto desta etapa:** `movies_feature_engineered.csv` — dataset analiticamente completo para os notebooks de análise

In [1]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..')))


# Importação de Bibliotecas e Módulos
import ast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Nossos módulos customizados da pasta src/
from src import load_data

---
## Carregando Dados

In [2]:
caminho = '../data/processed/movies_cleaned.pkl'
df_movies = load_data(caminho, tipo_arquivo='pkl')

Dados PKL carregados! Formato: (41164, 18)


---
## Correção de Franquias: Aplicação da Regra de Pareto ao Enriquecimento de Dados

A coluna `belongs_to_collection` do dataset original apresenta ausências para filmes de grandes franquias — não por ausência de vínculo, mas por limitações de cobertura da base de dados na época de sua criação. Como o pertencimento a uma franquia é uma das dimensões analíticas mais relevantes deste projeto, a ausência desses vínculos introduziria viés sistemático nas comparações entre filmes independentes e de franquia.

A correção prioriza as **franquias de maior impacto comercial e cultural**, seguindo o princípio de que um subconjunto reduzido de propriedades intelectuais representa a maior parte da receita de bilheteria global. O mapeamento explícito por palavra-chave no título garante que os filmes mais analiticamente relevantes estejam corretamente classificados.

- **Regra de negócio:** Somente registros com `belongs_to_collection` ausente recebem a correção — vínculos existentes no dataset original nunca são sobrescritos
- **Escopo da correção:** Franquias do universo Marvel, DC, grandes sagas de animação, ação e aventura que compõem o topo da distribuição de bilheteria histórica

In [3]:
# Dicionário Expandido: Mapeamento das Maiores Franquias do Cinema 
correcoes_franquias = {
    # 🦸‍♂️ Heróis (Marvel & DC)
    'Avengers': 'The Avengers Collection',
    'Vingadores': 'The Avengers Collection',
    'Spider-Man': 'Spider-Man Collection',
    'Homem-Aranha': 'Spider-Man Collection',
    'Iron Man': 'Iron Man Collection',
    'Homem de Ferro': 'Iron Man Collection',
    'Captain America': 'Captain America Collection',
    'Thor': 'Thor Collection',
    'X-Men': 'X-Men Collection',
    'Batman': 'Batman Collection',
    'Dark Knight': 'Batman Collection',
    'Cavaleiro das Trevas': 'Batman Collection',
    'Superman': 'Superman Collection',
    
    # 🧙‍♂️ Sci-Fi & Fantasia
    'Star Wars': 'Star Wars Collection',
    'Harry Potter': 'Harry Potter Collection',
    'Lord of the Rings': 'The Lord of the Rings Collection',
    'Senhor dos Anéis': 'The Lord of the Rings Collection',
    'Hobbit': 'The Hobbit Collection',
    'Matrix': 'The Matrix Collection',
    'Hunger Games': 'The Hunger Games Collection',
    'Jogos Vorazes': 'The Hunger Games Collection',
    'Twilight': 'The Twilight Saga Collection',
    'Crepúsculo': 'The Twilight Saga Collection',
    'Jumanji': 'Jumanji Collection',

    # 💥 Ação, Aventura & Sci-Fi
    'James Bond': 'James Bond Collection',
    '007': 'James Bond Collection',
    'Fast and Furious': 'The Fast and the Furious Collection',
    'Velozes e Furiosos': 'The Fast and the Furious Collection',
    'Mission: Impossible': 'Mission: Impossible Collection',
    'Missão: Impossível': 'Mission: Impossible Collection',
    'Indiana Jones': 'Indiana Jones Collection',
    'Pirates of the Caribbean': 'Pirates of the Caribbean Collection',
    'Piratas do Caribe': 'Pirates of the Caribbean Collection',
    'Jurassic': 'Jurassic Park Collection', 
    'Terminator': 'The Terminator Collection',
    'Exterminador do Futuro': 'The Terminator Collection',
    'Transformers': 'Transformers Collection',
    'Mad Max': 'Mad Max Collection',

    # 🎬 Animações Gigantes
    'Toy Story': 'Toy Story Collection',
    'Shrek': 'Shrek Collection',
    'Despicable Me': 'Despicable Me Collection',
    'Meu Malvado Favorito': 'Despicable Me Collection',
    'Minions': 'Despicable Me Collection',
    'Ice Age': 'Ice Age Collection',
    'A Era do Gelo': 'Ice Age Collection'
}

In [4]:
# Varrer o dicionário e aplicar a regra
sucessos = 0
for palavra_chave, nome_franquia in correcoes_franquias.items():
    # Encontra os filmes pelo padrão de texto no título
    filtro = df_movies['original_title'].str.contains(palavra_chave, case=False, na=False)
    
    # Conta quantos vão ser alterados para o log
    qtd_alterar = (filtro & (df_movies['belongs_to_collection'] == 'Filme Isolado')).sum()
    sucessos += qtd_alterar
    
    # Aplica a regra de negócio
    df_movies.loc[filtro & (df_movies['belongs_to_collection'] == 'Filme Isolado'), 
                  'belongs_to_collection'] = nome_franquia

print(f"Dicionário de {len(correcoes_franquias)} chaves processado.")
print(f"Total de filmes realocados para franquias: {sucessos}")

Dicionário de 45 chaves processado.
Total de filmes realocados para franquias: 93


---
## Checkpoint de Persistência: Separação entre Enriquecimento e Desnormalização

O dataset é salvo neste ponto — após o enriquecimento de features e antes de qualquer operação de explosão de colunas — para preservar a estrutura **um-filme-por-linha**. Essa separação é uma decisão arquitetural deliberada: análises financeiras e de atributos agregados do filme (orçamento, receita) consomem este artefato; análises por gênero ou país consumirão versões explodidas derivadas dele, mantendo a rastreabilidade entre as camadas do pipeline.

In [5]:
# Salvando o DataFrame processado para a próxima etapa de análise
caminho_saida = '../data/processed/movies_enriched.csv'
df_movies.to_csv(caminho_saida, index=False)

---
##  Modelagem Dimensional: Atomização de Listas Complexas

O dataset enriquecido armazena informações cruciais (gêneros, países de produção e idiomas) no formato de listas dentro de uma única célula. Embora eficiente para armazenamento, essa estrutura **inviabiliza análises categóricas isoladas**. Para responder a perguntas de negócio como "qual país produz os filmes mais rentáveis?" ou "qual o ROI médio do gênero terror?", é imperativo normalizar essas variáveis para um modelo *one-to-many*.

- **Decisão arquitetural (Prevenção de Produto Cartesiano):** Optou-se por **não** explodir múltiplas colunas no mesmo *dataframe*. Fazer isso geraria uma multiplicação exponencial das linhas (ex: Gênero $\times$ País $\times$ Idioma), o que corromperia gravemente o cálculo de métricas financeiras como `revenue` e `budget`. Em vez disso, adotamos a criação de **tabelas dimensão independentes** para cada categoria.
- **Trade-off aceito:** O projeto passará a gerenciar múltiplos arquivos `.csv` (uma base principal estática e bases dimensionais explodidas) em detrimento de uma tabela única universal. Essa fragmentação intencional blinda a integridade estatística dos dados globais enquanto habilita mergulhos analíticos profundos em categorias específicas.
---

### 1. Dimensão de Gêneros (`movies_genres`)

Isolamento da variável `genres` para viabilizar análises de volume e performance financeira por categoria cinematográfica. 

* **Operação:** Fatiamento das listas e aplicação do `.explode()`.
* **Validação:** Inspeção da nova granularidade (múltiplas linhas por filme) e integridade dos tipos.
* **Persistência:** Exportação da tabela dimensão para `data/processed/movies_genres.csv`.

In [6]:
# Explodindo a coluna de gêneros para criar uma linha por gênero
df_genres_exploded = df_movies.explode('genres')

# Limpando os espaços em branco dos gêneros
df_genres_exploded['genres'] = df_genres_exploded['genres'].str.strip()

# Resetando o índice para evitar índices duplicados
df_genres_exploded.reset_index(drop=True, inplace=True)

In [7]:
# Validação da granularidade atômica: uma linha por par filme-gênero
display(
    df_genres_exploded[['original_title', 'genres']]
    .head(20)
    .style
    .set_properties(
        subset=['original_title'],
        **{'background-color': '#2B2D42', 'color': 'white'}
    )
    .set_properties(
        subset=['genres'],
        **{'background-color': '#6D597A', 'color': 'white'}
    )
)

,original_title,genres
0,Toy Story,Animation
1,Toy Story,Comedy
2,Toy Story,Family
3,Jumanji,Adventure
4,Jumanji,Fantasy
5,Jumanji,Family
6,Grumpier Old Men,Romance
7,Grumpier Old Men,Comedy
8,Waiting to Exhale,Comedy
9,Waiting to Exhale,Drama


In [8]:
# Verificando o número de linhas após a explosão
display(f"Número de linhas após a explosão: {df_genres_exploded.shape[0]}")

'Número de linhas após a explosão: 87072'

### Salvando dados após a explosão da coluna `genres` 

In [9]:
# Salvando o DataFrame explodido para análise de gêneros
caminho_saida_genres_exploded = '../data/processed/movies_genres.csv'
df_genres_exploded.to_csv(caminho_saida_genres_exploded, index=False)

### 2. Dimensão de Países de Produção (`movies_countries`)

Extração da variável `production_countries` para viabilizar o mapeamento geográfico da indústria e análises comparativas de performance financeira por origem de produção — como a distinção entre Hollywood, Bollywood e cinematografias europeias.

- **Motivação analítica:** A co-produção internacional é um fenômeno relevante no cinema moderno; a estrutura explodida permite quantificar com precisão a participação de cada país, inclusive em produções compartilhadas
- **Persistência:** Exportação da tabela dimensão para `data/processed/movies_countries.csv`

In [10]:
# Explodindo a coluna de países para criar uma linha por país
df_countries_exploded = df_movies.explode('production_countries')

# Limpando os espaços em branco dos países
df_countries_exploded['production_countries'] = df_countries_exploded['production_countries'].str.strip()

# Resetando o índice para evitar índices duplicados
df_countries_exploded.reset_index(drop=True, inplace=True)

In [11]:
# Validação da granularidade atômica: uma linha por par filme-país
display(
    df_countries_exploded[['original_title', 'production_countries']]
    .head(20)
    .style
    .set_properties(
        subset=['original_title'],
        **{'background-color': '#2B2D42', 'color': 'white'}
    )
    .set_properties(
        subset=['production_countries'],
        **{'background-color': '#588157', 'color': 'white'}
    )
)

,original_title,production_countries
0,Toy Story,United States of America
1,Jumanji,United States of America
2,Grumpier Old Men,United States of America
3,Waiting to Exhale,United States of America
4,Father of the Bride Part II,United States of America
5,Heat,United States of America
6,Sabrina,Germany
7,Sabrina,United States of America
8,Tom and Huck,United States of America
9,Sudden Death,United States of America


In [12]:
# Verificando o número de linhas após a explosão
display(f"Número de linhas após a explosão: {df_countries_exploded.shape[0]}")

'Número de linhas após a explosão: 50958'

### Salvando dados após a explosão da coluna `production_countries` 

In [13]:
# Salvando o DataFrame explodido para análise de países
caminho_saida_countries_exploded = '../data/processed/movies_countries.csv'
df_countries_exploded.to_csv(caminho_saida_countries_exploded, index=False)

### 3. Dimensão de Idiomas Falados (`movies_languages`)

Separação da variável `spoken_languages` para habilitar análises de alcance geolinguístico e correlação entre diversidade de idiomas e performance de bilheteria internacional. Filmes multilíngues têm potencial de distribuição estruturalmente diferente dos produziods para mercados monolíngues.

- **Motivação analítica:** O idioma é um proxy de mercado-alvo — identificar padrões entre o número de idiomas de um filme e sua receita global é uma hipótese central do projeto
- **Persistência:** Exportação da tabela dimensão para `data/processed/movies_languages.csv`

In [14]:
# Explodindo a coluna de idiomas para criar uma linha por idioma
df_languages_exploded = df_movies.explode('spoken_languages')

# Limpando os espaços em branco dos idiomas
df_languages_exploded['spoken_languages'] = df_languages_exploded['spoken_languages'].str.strip()

# Resetando o índice para evitar índices duplicados
df_languages_exploded.reset_index(drop=True, inplace=True)

In [15]:
# Validação da granularidade atômica: uma linha por par filme-idioma
display(
    df_languages_exploded[['original_title', 'spoken_languages']]
    .head(20)
    .style
    .set_properties(
        subset=['original_title'],
        **{'background-color': '#2B2D42', 'color': 'white'}
    )
    .set_properties(
        subset=['spoken_languages'],
        **{'background-color': '#3D5A80', 'color': 'white'}
    )
)

,original_title,spoken_languages
0,Toy Story,English
1,Jumanji,English
2,Jumanji,Français
3,Grumpier Old Men,English
4,Waiting to Exhale,English
5,Father of the Bride Part II,English
6,Heat,English
7,Heat,Español
8,Sabrina,Français
9,Sabrina,English


In [16]:
# Verificando o número de linhas após a explosão
display(f"Número de linhas após a explosão: {df_languages_exploded.shape[0]}")

'Número de linhas após a explosão: 52433'

### Salvando dados após a explosão da coluna `spoken_languages`

In [17]:
# Salvando o DataFrame explodido para análise de idiomas
caminho_saida_languages_exploded = '../data/processed/movies_languages.csv'
df_languages_exploded.to_csv(caminho_saida_languages_exploded, index=False)

### 4. Dimensão de Produtoras (`movies_companies`)

Separação da variável `production_companies` para habilitar análises de *market share*, poder de distribuição e correlação entre o selo do estúdio e o sucesso financeiro (bilheteria e ROI). Obras financiadas por grandes conglomerados da indústria (*Majors*) apresentam comportamentos de risco e escalabilidade estruturalmente diferentes de produções independentes.

- **Motivação analítica:** O estúdio produtor é um forte indicativo da capacidade de investimento e alcance de marketing. Identificar quais empresas dominam gêneros específicos, possuem as maiores margens de lucro ou concentram o controle das grandes franquias é um dos pilares de inteligência de negócio deste projeto.
- **Persistência:** Exportação da tabela dimensão para `data/processed/movies_companies.csv`.

In [18]:
# Explodindo a coluna de produtoras para criar uma linha por produtora
df_companies_exploded = df_movies.explode('production_companies')

# Limpando os espaços em branco
df_companies_exploded['production_companies'] = df_companies_exploded['production_companies'].str.strip()

# Resetando o índice para evitar índices duplicados
df_companies_exploded.reset_index(drop=True, inplace=True)

In [19]:
# Validação da granularidade atômica: uma linha por par filme-produtora
display(
    df_companies_exploded[['original_title', 'production_companies']]
    .head(20)
    .style
    .set_properties(
        subset=['original_title'],
        **{'background-color': '#2B2D42', 'color': 'white'}
    )
    .set_properties(
        subset=['production_companies'],
        **{'background-color': '#4A4E69', 'color': 'white'}
    )
)

,original_title,production_companies
0,Toy Story,Pixar Animation Studios
1,Jumanji,TriStar Pictures
2,Jumanji,Teitler Film
3,Jumanji,Interscope Communications
4,Grumpier Old Men,Warner Bros.
5,Grumpier Old Men,Lancaster Gate
6,Waiting to Exhale,Twentieth Century Fox Film Corporation
7,Father of the Bride Part II,Sandollar Productions
8,Father of the Bride Part II,Touchstone Pictures
9,Heat,Regency Enterprises


In [20]:
# Verificando o número de linhas após a explosão
display(f"Número de linhas após a explosão: {df_companies_exploded.shape[0]}")

'Número de linhas após a explosão: 77089'

### Salvando dados após a explosão da coluna `production_companies`

In [21]:
# Salvando o DataFrame explodido para análise de produtoras
caminho_saida_companies_exploded = '../data/processed/movies_companies.csv'
df_companies_exploded.to_csv(caminho_saida_companies_exploded, index=False)